# Introduction: Ollama LLM Backend

Ollama is a powerful open-source platform for running large language models (LLMs) locally on your machine. It operates as an independent service with a built-in server that exposes an OpenAI-compatible API endpoint at http://localhost:11434/v1.

This makes Ollama perfect for AI agent development, local experimentation, and production deployments without cloud dependencies. You'll download models once, then query them through familiar OpenAI SDK calls — but everything runs offline on your hardware.
Why Ollama?

- Local-first: No API costs, no data sent to third parties
- OpenAI compatible: Use existing Python SDKs and agent frameworks
- Model management: Easy pull/list/ps commands for model lifecycle
- Performance optimized: GPU acceleration, streaming, context caching
- Developer friendly: REST API + CLI in one package

The great thing about Ollama being OpenAI compatible is that we can build code using OpenAI and locally hosted models, but then switch to high performing models like those hosted on tejas.   In most cases, we will only need to change our base_url and api_keys.  

If you havent already you can download Ollama by following instructions here: https://ollama.com/download/windows 

## Ollama CLI interface 

Here are a few commonly used Ollama CLI commands to download, explore and use models. 
```bash 
# list models downloaded on your machine 
ollama list            

# download an additional model 
ollama pull qwen3     

# launches the local Ollama API/server, usually on port 11434. Can chat programattically with any model
ollama serve  
```

Additionally you can chat with models via command line.  Here are some additional useful commands for that 
```bash
# makes sure the model is available, then opens an interactive chat session with it through the server.
ollama run qwen3    

# checks which models are running 
ollama ps 

# stops model from running when you are no long using
ollama stop qwen3
```

We will focus on using Ollama programmatically in this tutorial. Let's first chat with an Ollama model with OpenAI. First we will use commands from above to make sure we have the correct models downloaded and local server started:

In [ ]:
!ollama list 

In [ ]:
#!ollama pull llama3.2:3b

In [ ]:
#!ollama serve

Now let's send a message the llamma3b with openai 

In [ ]:
# Import OpenAI client class
from openai import OpenAI

In [ ]:
# define variable for local model we will use 
llama3b = "llama3.2:3b"

# connect to ollama running in the backend 
openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# Define a wrapper function that will send requests to an LLM and receive responses
def generate_response(
        client: OpenAI,
        model: str,
        user_prompt: str,                                      # Most important text to send; typical prompt we send LLMs
        system_prompt: str = "You are a helpful assistant.",   # Describe AI's overall role, behavior, tone, and constraints across all interactions
        temperature: float = 0.0       
) -> str:
    """Sends a request to an LLM and and returns a response."""
    
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

# Vague prompt = vague result.
# We should expect a generic output from the LLM.
vague_prompt = "Tell me briefly about Albert Einstein."

vague_response = generate_response(
    client=openai_client,
    model=llama3b,
    user_prompt=vague_prompt, 
)

print(vague_response)

### Samba Nova Set Up

We can easily use the same function from above to chat with models available on tejas. Let's use this to check which models are available with OpenAI.

Before we do that we will need to safely import the tejas api key.  We will use the dotenv library and will need to do the following:

1. Create a file in the current directory called .env
2. Add one line to this file:

```bash 
API_KEY="insert-tejas-api-key"
```

In [ ]:
# samba base url 
samba_base_url = "https://ai.tejas.tacc.utexas.edu"

# get api key from hidden file 
from dotenv import load_dotenv
import os 
load_dotenv()  # Loads .env into os.environ
samba_api_key = os.getenv("API_KEY") 

In [ ]:
# connect to tejas
openai_client_samba = OpenAI(
    base_url=samba_base_url,
    api_key=samba_api_key
)

In [ ]:
# check what models are available 
models = openai_client_samba.models.list()
for model in models.data:
    print(f"ID: {model.id}, Owned by: {model.owned_by}")

Next we can send a message to tejas with the same functions generated for Ollama code. 

In [ ]:
# define variable for local model we will use 
llama4 = "Llama-4-Maverick-17B-128E-Instruct"  # MOE model: 17B parameters used in inference and 128 Experts

# send a message to samba nova 
vague_response = generate_response(
    client=openai_client_samba,
    model=llama4,
    user_prompt=vague_prompt, 
)

print(vague_response)

## Importing Academic Papers and Summarizing with LLMs 

In [ ]:
!ls docs2

In [ ]:
from pypdf import PdfReader

def read_pdf(path: str) -> str:
    reader = PdfReader(path)
    return "".join(page.extract_text() or "" for page in reader.pages)

# Read paper
pdf_text = read_pdf("docs2/manuscript_foundation_model_for_earth_system.pdf")

# Inject into prompt (example)
prompt = f"""You are reviewing a research paper.
Here is the paper text:

{pdf_text}

Summarize the main contributions and methodology."""

summary = generate_response(
    client=openai_client_samba,
    model=llama4,
    user_prompt=prompt, 
)

print(summary)

## Extracting Meta Data with LLMs 

First PDFs do come with metadata that you can grab as seen in the simple code demo below:

In [ ]:
# first PDFs do come with metadata that you can grab.  
reader = PdfReader("docs2/manuscript_foundation_model_for_earth_system.pdf")
reader.metadata

However, there is likely paper specific data that you would like to obtain that is not included in the meta data above that may require reading the entire paper.  Thankfully, LLMs are pretty good and extracting this information particularly if you have a very specific factual inquiry.  Next, we will demo how we can get LLMs to extract this information from paper.

### Structured Generation

One of the biggest challenges when working with large language models (LLMs) is that they typically produce unstructured text—free-form answers that might not follow a consistent or machine-readable format. Structured generation solves this problem by defining a schema (a data structure the model must follow) and constraining the output to match it. This is particularly useful for applications that require reliability and consistency, such as agents, tools, or pipelines that process AI-generated data automatically.

To implement structured generation you will need to do a few things with new Python libraries: 

1. Wrap the openai client with **instructor** library: The from_openai function from instructor bridges the OpenAI client and structured output generation. It automatically prompts the model with a json constraint and validates the response against a schema.
2. Use **Pydantic** to define a schema for the output of an llm. 

In [ ]:
import instructor
from pydantic import BaseModel, Field
import json

instructor_client = instructor.from_openai(
    openai_client_samba,
    mode=instructor.Mode.JSON # When using with ollama, provide mode parameter 
)

# pydantic schema describing json data we would like returned
class MetaData(BaseModel):
    journal: str = Field(..., description="The journal the article was published in.")
    last_author: str = Field(..., description="The first and last name of the last author listed")

# prompt templates for extracting meta data
system_prompt = "You are a meta data extractor."
user_prompt = f"""Extract the meta data as specified in the data schema provided from the following academic paper: 
                
                {pdf_text}
                """

# set this request to the sambanova
response = instructor_client.chat.completions.create(
    model=llama4,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_model=MetaData,
    max_retries=3,
)

# Convert Pydantic model to Python dictionary 
json_response = response.model_dump()

print(json_response)

## Extracting Meta Data from Multiple Papers  

First we will create a function that reads in all papers in the folder docs2 and stores the text of the paper in a dictionary.  

In [ ]:
from pathlib import Path
from pypdf import PdfReader

def read_pdfs_from_dir(dir_path="docs2/"):
    pdf_dir = Path(dir_path)
    papers = {}

    for pdf_path in sorted(pdf_dir.glob("*.pdf")):
        reader = PdfReader(str(pdf_path))
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        papers[pdf_path.name] = text

    return papers

papers = read_pdfs_from_dir("docs2/")
print(f"Loaded {len(papers)} PDFs")

Then we loop through every paper and do one LLM structured generation call to extract meta data from papers.  We append the results from every paper into a table. 

In [ ]:
class MetaData(BaseModel):
    journal: str = Field(..., description="The journal the article was published in.")
    first_author: str = Field(...,  description="The first and last name of the first author listed")
    last_author: str = Field(..., description="The first and last name of the last author listed")
    title: str = Field(..., description="The title of the research article")
    
json_all = []
for name, paper in papers.items():
    user_prompt = f"""Extract the meta data as specified in the data schema provided from the follwoing academic paper: 
                
                {paper}
                """
    response = instructor_client.chat.completions.create(
        model=llama4,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_model=MetaData,
        max_retries=3,
    )
    
    # Convert Pydantic model to Python dictionary 
    json_response = response.model_dump()

    # append extracted data to create table
    json_all.append(json_response) 

In [ ]:
json_all

### Finally we can write the table we created to a csv file.

In [ ]:
import pandas as pd 

In [ ]:
df = pd.DataFrame(json_all)
df

In [ ]:
df.to_csv("meta_data.csv")

In [ ]:
!ls

# Last Note: Evaluation of LLM pipelines 

Extracting Meta Data using LLMs is going to be handy as you build out an increasing large database of papers.  But, LLMs aren't perfect and they will occasionally halucinate information.  

However, we can evaluate our pipelines by comparing the LLM generated results with known solutions and use this feedback to optimize performance by changing things like prompt templates, schema descriptions, and models used.  This will help to keep hallucinations to a minimum. 